In [ ]:
%py
spark.catalog.setCurrentCatalog("purgo_databricks")

# PySpark script for comprehensive test coverage of compound screening analysis aggregation and categorization
# Purpose: Validate all transformation, aggregation, join, and categorization logic for compound drug analysis
# Author: Giang Nguyen
# Date: 2025-10-13
# Description: This script tests the extraction, filtering, aggregation, joining, scoring, and categorization of compound drug analysis data. It includes schema validation, data type checks, error handling, and asserts for all critical business logic, including NULL handling and complex type validation.

# Required imports for PySpark DataFrame operations and testing
from pyspark.sql import DataFrame 
from pyspark.sql import functions as F 
from pyspark.sql.types import ( 
    StructType, StructField, StringType, DoubleType, LongType, IntegerType
)
from pyspark.sql.utils import AnalysisException 

# -- Test utility: Assert DataFrame schema matches expected schema
def assert_schema(df: DataFrame, expected_schema: StructType) -> None:
    """
    Assert that the DataFrame schema matches the expected schema.

    Args:
        df (DataFrame): The DataFrame to check.
        expected_schema (StructType): The expected schema.

    Returns:
        None
    """
    actual_fields = [(f.name, f.dataType, f.nullable) for f in df.schema.fields]
    expected_fields = [(f.name, f.dataType, f.nullable) for f in expected_schema.fields]
    assert actual_fields == expected_fields, f"Schema mismatch: {actual_fields} != {expected_fields}"

# -- Test utility: Assert DataFrame row count matches expected
def assert_row_count(df: DataFrame, expected_count: int) -> None:
    """
    Assert that the DataFrame row count matches the expected count.

    Args:
        df (DataFrame): The DataFrame to check.
        expected_count (int): The expected row count.

    Returns:
        None
    """
    actual_count = df.count()
    assert actual_count == expected_count, f"Row count mismatch: {actual_count} != {expected_count}"

# -- Test utility: Assert DataFrame columns match expected columns
def assert_columns(df: DataFrame, expected_columns: list) -> None:
    """
    Assert that the DataFrame columns match the expected columns.

    Args:
        df (DataFrame): The DataFrame to check.
        expected_columns (list): The expected column names.

    Returns:
        None
    """
    actual_columns = df.columns
    assert actual_columns == expected_columns, f"Column mismatch: {actual_columns} != {expected_columns}"

# -- Test utility: Assert values in a DataFrame column match expected values
def assert_column_values(df: DataFrame, column: str, expected_values: list) -> None:
    """
    Assert that the values in a DataFrame column match the expected values.

    Args:
        df (DataFrame): The DataFrame to check.
        column (str): The column name.
        expected_values (list): The expected values.

    Returns:
        None
    """
    actual_values = [row[column] for row in df.select(column).collect()]
    assert actual_values == expected_values, f"Column values mismatch for {column}: {actual_values} != {expected_values}"

# -- Define expected schema for purgo_playground.compound_drug_analysis
compound_schema = StructType([
    StructField("study_id", StringType(), True),
    StructField("compound_id", StringType(), True),
    StructField("mutation_id", StringType(), True),
    StructField("therapeutic_area", StringType(), True),
    StructField("drug_name", StringType(), True),
    StructField("ic50", DoubleType(), True),
    StructField("auc", DoubleType(), True),
    StructField("efficacy", DoubleType(), True),
    StructField("toxicity", DoubleType(), True),
    StructField("potency", DoubleType(), True),
    StructField("sample_size", LongType(), True),
    StructField("mutation_frequency", LongType(), True),
    StructField("mutation_severity", LongType(), True),
    StructField("compound_concentration", DoubleType(), True),
    StructField("cell_viability", DoubleType(), True),
    StructField("growth_inhibition", DoubleType(), True),
    StructField("result", StringType(), True),
    StructField("approved_flag", LongType(), True),
    StructField("validation_status", StringType(), True),
    StructField("status", StringType(), True),
    StructField("created_by", StringType(), True),
    StructField("score1", DoubleType(), True),
    StructField("score2", DoubleType(), True),
    StructField("score3", DoubleType(), True),
    StructField("score4", DoubleType(), True),
    StructField("score5", DoubleType(), True)
])

# -- Read compound_drug_analysis table with error handling for missing or invalid data
try:
    compound_df = spark.table("purgo_databricks.purgo_playground.compound_drug_analysis")
except AnalysisException as e:
    compound_df = spark.createDataFrame([], compound_schema)
    print(f"Error reading compound_drug_analysis: {e}")

# -- Validate schema of loaded DataFrame
assert_schema(compound_df, compound_schema)

# -- Filter Analysis: Only rows with approved_flag == 1 and validation_status == 'valid'
filtered_df = compound_df.filter(
    (F.col("approved_flag") == 1) & (F.col("validation_status") == "valid")
)

# -- Assert filtering logic: Only expected rows included
assert_row_count(filtered_df, 2) # S001 and S002 only

# -- Aggregation: Group by therapeutic_area and calculate metrics
agg_df = compound_df.filter(
    (F.col("approved_flag") == 1) & (F.col("validation_status") == "valid")
).groupBy("therapeutic_area").agg(
    F.avg("ic50").alias("avg_ic50"),
    F.avg("auc").alias("avg_auc"),
    F.avg("efficacy").alias("avg_efficacy"),
    F.sum("sample_size").alias("total_sample_size"),
    F.countDistinct("study_id").alias("study_count")
)

# -- Assert aggregation results for Oncology and Cardiology
oncology_row = agg_df.filter(F.col("therapeutic_area") == "Oncology").collect()
cardiology_row = agg_df.filter(F.col("therapeutic_area") == "Cardiology").collect()
assert len(oncology_row) == 1
assert len(cardiology_row) == 1
assert abs(oncology_row[0]["avg_ic50"] - 0.45) < 1e-6
assert abs(cardiology_row[0]["avg_ic50"] - 1.20) < 1e-6

# -- Join Analysis: Join filtered data with aggregated metrics on therapeutic_area
joined_df = filtered_df.join(
    agg_df,
    on="therapeutic_area",
    how="left"
)

# -- Assert join: All filtered rows have correct aggregated metrics
assert_row_count(joined_df, 2)
assert_columns(
    joined_df,
    [
        "therapeutic_area", "study_id", "compound_id", "mutation_id", "drug_name", "ic50", "auc", "efficacy",
        "toxicity", "potency", "sample_size", "mutation_frequency", "mutation_severity", "compound_concentration",
        "cell_viability", "growth_inhibition", "result", "approved_flag", "validation_status", "status", "created_by",
        "score1", "score2", "score3", "score4", "score5", "avg_ic50", "avg_auc", "avg_efficacy", "total_sample_size", "study_count"
    ]
)

# -- Result Analysis: Compute overall_score and categorize results
def compute_overall_score(row) -> float:
    """
    Compute the average of score1 to score5 for a row.

    Args:
        row (Row): A Spark Row object.

    Returns:
        float: The average score, or None if any score is missing or invalid.
    """
    scores = [row["score1"], row["score2"], row["score3"], row["score4"], row["score5"]]
    if any(s is None or not isinstance(s, float) for s in scores):
        return None
    return float(sum(scores)) / 5.0

def categorize_potential(overall_score: float) -> str:
    """
    Categorize the potential based on overall_score.

    Args:
        overall_score (float): The overall score.

    Returns:
        str: The potential category.
    """
    if overall_score is None:
        return "Unknown"
    if 70 <= overall_score <= 100:
        return "High Potential"
    elif 60 <= overall_score < 70:
        return "Moderate Potential"
    elif overall_score < 60:
        return "Low Potential"
    else:
        return "Unknown"

# -- Add overall_score and potential_category columns
joined_df = joined_df.withColumn(
    "overall_score",
    F.when(
        F.col("score1").isNull() | F.col("score2").isNull() | F.col("score3").isNull() | F.col("score4").isNull() | F.col("score5").isNull(),
        F.lit(None)
    ).otherwise(
        (F.col("score1") + F.col("score2") + F.col("score3") + F.col("score4") + F.col("score5")) / 5.0
    )
).withColumn(
    "potential_category",
    F.when(F.col("overall_score").isNull(), F.lit("Unknown"))
     .when((F.col("overall_score") >= 70) & (F.col("overall_score") <= 100), F.lit("High Potential"))
     .when((F.col("overall_score") >= 60) & (F.col("overall_score") < 70), F.lit("Moderate Potential"))
     .when(F.col("overall_score") < 60, F.lit("Low Potential"))
     .otherwise(F.lit("Unknown"))
)

# -- Assert scoring and categorization logic
score_rows = joined_df.select("study_id", "overall_score", "potential_category").collect()
for row in score_rows:
    if row["study_id"] == "S001":
        assert abs(row["overall_score"] - 82.0) < 1e-6
        assert row["potential_category"] == "High Potential"
    elif row["study_id"] == "S002":
        assert abs(row["overall_score"] - 65.0) < 1e-6
        assert row["potential_category"] == "Moderate Potential"

# -- Data Type Testing: Validate approved_flag is integer and NULL handling
invalid_flag_df = compound_df.filter(F.col("approved_flag").isNull())
assert_row_count(invalid_flag_df, 1) # S005

# -- Data Quality Validation: Ensure no column mismatch before insert
expected_output_columns = [
    "study_id", "compound_id", "mutation_id", "therapeutic_area", "drug_name", "ic50", "auc", "efficacy",
    "toxicity", "potency", "sample_size", "mutation_frequency", "mutation_severity", "compound_concentration",
    "cell_viability", "growth_inhibition", "result", "approved_flag", "validation_status", "status", "created_by",
    "score1", "score2", "score3", "score4", "score5", "avg_ic50", "avg_auc", "avg_efficacy", "total_sample_size", "study_count",
    "overall_score", "potential_category"
]
assert_columns(joined_df.select(*expected_output_columns), expected_output_columns)

# -- Performance Test: Ensure aggregation and join are efficient for batch scenario
import time 
start_time = time.time()
_ = compound_df.filter(
    (F.col("approved_flag") == 1) & (F.col("validation_status") == "valid")
).groupBy("therapeutic_area").agg(
    F.avg("ic50").alias("avg_ic50"),
    F.avg("auc").alias("avg_auc"),
    F.avg("efficacy").alias("avg_efficacy"),
    F.sum("sample_size").alias("total_sample_size"),
    F.countDistinct("study_id").alias("study_count")
).collect()
duration = time.time() - start_time
assert duration < 10, f"Performance test failed: aggregation took {duration} seconds"

# -- Delta Lake Operations: Test MERGE, UPDATE, DELETE on result table
from delta.tables import DeltaTable 

# -- Setup: Create a Delta table for testing if not exists
result_table_name = "purgo_databricks.purgo_playground.compound_screening_result_analysis"
try:
    result_delta = DeltaTable.forName(spark, result_table_name)
except AnalysisException:
    joined_df.limit(0).write.format("delta").saveAsTable(result_table_name)
    result_delta = DeltaTable.forName(spark, result_table_name)

# -- MERGE: Upsert new results
merge_source_df = joined_df.select(*expected_output_columns)
result_delta.alias("tgt").merge(
    merge_source_df.alias("src"),
    "tgt.study_id = src.study_id"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

# -- UPDATE: Set potential_category to 'Low Potential' for overall_score < 60
result_delta.update(
    condition="overall_score < 60",
    set={"potential_category": "'Low Potential'"}
)

# -- DELETE: Remove rows with NULL overall_score
result_delta.delete("overall_score IS NULL")

# -- Window Function Test: Rank compounds by overall_score within each therapeutic_area
from pyspark.sql.window import Window 
window_spec = Window.partitionBy("therapeutic_area").orderBy(F.desc("overall_score"))
ranked_df = joined_df.withColumn("score_rank", F.rank().over(window_spec))
assert "score_rank" in ranked_df.columns

# -- Complex Type Validation: Test ARRAY and STRUCT creation
array_df = joined_df.withColumn("score_array", F.array("score1", "score2", "score3", "score4", "score5"))
struct_df = joined_df.withColumn("score_struct", F.struct("score1", "score2", "score3", "score4", "score5"))
assert "score_array" in array_df.columns
assert "score_struct" in struct_df.columns

# -- NULL Handling Test: Ensure rows with NULL scores are handled
null_score_df = compound_df.filter(
    F.col("score1").isNull() | F.col("score2").isNull() | F.col("score3").isNull() | F.col("score4").isNull() | F.col("score5").isNull()
)
assert_row_count(null_score_df, 1) # S005

# -- Cleanup: Remove test rows from result table
result_delta.delete("study_id IN ('S001', 'S002', 'S005')")

# -- Final Output: Display results with all columns and result analysis
final_output_df = joined_df.select(*expected_output_columns)
final_output_df.show()

# -- End of script
